# Train VPO on the maze task (Colab T4 + LoRA)

<a href="https://colab.research.google.com/github/ryanboldi/vpo/blob/main/notebooks/03_train_maze_colab.ipynb"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

> **Status: draft / experimental.** End-to-end training on a free Colab T4 with LoRA on a 0.5B model.
> Tight on memory; you may need to drop `train_batch_size` or `rollout.n` further if you OOM.
> Recommended: switch the Colab runtime to **GPU → T4**.

This notebook runs the full VPO pipeline:
1. Clone the repo and install patched veRL.
2. Preprocess the maze task (synthetic — no HF download).
3. Train VPO with LoRA on a tiny model for one epoch.
4. Plot the in-loop validation curve.

Expected wall-clock: **~30-45 minutes** on a free Colab T4.


## 1. Clone + install


In [ ]:
!git clone https://github.com/ryanboldi/vpo.git
%cd vpo


In [ ]:
# Install patched veRL first (it owns the heavy deps).
!cd verl && pip install -q -e . && cd ..

# Then the vpo package.  vllm==0.12.0 is pinned in pyproject.
!pip install -q -e .

# Verify the patched veRL has VPO enums.
!python -c "from verl.trainer.ppo.core_algos import AdvantageEstimator as AE; print([AE[n].value for n in ['VPO','VPO_SINGLE','MAXRL']])"


## 2. Preprocess maze (synthetic — no HF download)


In [ ]:
!python data/preprocess_maze.py --local_save_dir /content/data/maze
!ls -la /content/data/maze/


## 3. Train VPO (LoRA, Qwen2.5-0.5B, 1 epoch, 1 GPU)

LoRA + tiny batch keeps memory under 14 GB. If you hit OOM, drop `rollout.n` from 4 to 2 or
`max_response_length` from 512 to 384.


In [ ]:
# Override the defaults baked into train.sh for Colab's T4.
import os
os.environ['MAZE_DATA']  = '/content/data/maze'
os.environ['MODEL']      = 'Qwen/Qwen2.5-0.5B-Instruct'
os.environ['N_GPUS']     = '1'
os.environ['EPOCHS']     = '1'
os.environ['TAG']        = 'colab_smoke'

# Forwarded Hydra overrides — keep memory in check.
HYDRA_ARGS = [
    '++data.train_batch_size=8',
    '++actor_rollout_ref.actor.ppo_mini_batch_size=4',
    '++actor_rollout_ref.actor.ppo_micro_batch_size_per_gpu=1',
    '++actor_rollout_ref.rollout.n=4',
    '++actor_rollout_ref.rollout.gpu_memory_utilization=0.55',
    '++data.max_response_length=512',
    '++actor_rollout_ref.model.lora_rank=16',
    '++actor_rollout_ref.model.lora_alpha=32',
    '++trainer.save_freq=20',
    '++trainer.test_freq=20',
]

cmd = 'bash train.sh METHOD=vpo TASK=maze ' + ' '.join(HYDRA_ARGS)
print(cmd)
!{cmd}


## 4. Plot the VPO diagnostic curve

The patched veRL emits per-step VPO diagnostics (`vpo/own_pool_expected_max_mean`,
`vpo/group_std_mean`, ...) to wandb. We can also tail them from the log file.


In [ ]:
import re, json
from pathlib import Path
import matplotlib.pyplot as plt

log = sorted(Path('logs').glob('vpo_maze_*.log'))[-1]
print(f'reading {log}')

# veRL logs metrics as JSON-ish lines in console output; this is a crude grep.
steps, expmax, gstd = [], [], []
for line in log.read_text().splitlines():
    m = re.search(r'step:\s*(\d+).*own_pool_expected_max_mean:\s*([\d.]+).*group_std_mean:\s*([\d.]+)', line)
    if m:
        steps.append(int(m.group(1)))
        expmax.append(float(m.group(2)))
        gstd.append(float(m.group(3)))

fig, axes = plt.subplots(1, 2, figsize=(11, 3.5))
axes[0].plot(steps, expmax, 'o-'); axes[0].set_title('vpo/own_pool_expected_max_mean (↑)')
axes[0].set_xlabel('step'); axes[0].grid(alpha=0.3)
axes[1].plot(steps, gstd, 'o-');  axes[1].set_title('vpo/group_std_mean (signal magnitude)')
axes[1].set_xlabel('step'); axes[1].grid(alpha=0.3)
plt.tight_layout(); plt.show()


## What just happened?

You trained a 0.5B model with VPO on the maze task. The `own_pool_expected_max_mean` curve should
trend upward — the model is learning to produce m=3 routes whose **best-of-m under random preferences**
is higher than the prompt-group baseline.

To run the real paper config (Qwen3-4B, full batch, 50 epochs), use a 4×H100 node and the bare command:
```bash
bash train.sh METHOD=vpo TASK=maze
```

That uses the defaults in `train.sh` — same hyperparameters as the paper. See
[`docs/reproducibility.md`](../docs/reproducibility.md) for the full configs.
